In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
from types import SimpleNamespace
import utils as course_utils
import unittests  # the solution checks of this notebook: unittests.py

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Device of the full run. 'auto' (the default) takes the first available of
# 'cuda' (NVIDIA GPU), 'xla' (TPU), 'mps' (Apple GPU, only when Python runs on
# macOS itself) and 'cpu'; the cell prints the one it took. The small run
# always uses the CPU.
DEVICE = 'auto'
# Settings of the full run (IS_UNITTEST = False). Free to change.
FULL_RUN = dict(
    TRAINING_EPOCHS=3,          # the training of Task 5
    EXPERIMENT_EPOCHS=100,      # every experiment of Task 6
)
# Settings of the small run. Keep them: grading compares your results with
# the teacher's results of exactly these settings.
SMALL_RUN = dict(
    TRAINING_EPOCHS=1,
    EXPERIMENT_EPOCHS=1,
)
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# The settings of this run, SMALL_RUN or FULL_RUN, as one namespace:
# run_params.TRAINING_EPOCHS, run_params.EXPERIMENT_EPOCHS.
run_params = SimpleNamespace(**(SMALL_RUN if IS_UNITTEST else FULL_RUN))
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)


seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

# Importing the source code
import source
src = source.code.src

# Preparing the answer collection
answer = {}

# The device of the training: always the CPU in the small run, DEVICE otherwise.
device = src.utils.device.select_device('cpu' if IS_UNITTEST else DEVICE)
print(f"Training device: {src.utils.device.describe(device)}"
      + (" (the small run always uses the CPU)" if IS_UNITTEST else f" (DEVICE = {DEVICE!r})"))
print("Run settings:", run_params)


# Task 0: Prepare the environment

<details>
<summary>What to do</summary>

## Goal

To make the experiment reproducible, implement `seed_all` in [src/utils/seed.py](../../source/code/src/utils/seed.py).

You can also use f12 key to navigate to the function definition.

## What To Seed

Inside this function, seed the random number generators used by:
- `random`
- `numpy`
- `torch`

## PyTorch Determinism

For PyTorch, seed the generator of every device the script may run on:
- CPU: `torch.manual_seed`;
- every NVIDIA GPU, not only the current one: `torch.cuda.manual_seed_all`;
- Apple GPU (Metal) on a MacBook: `torch.mps.manual_seed`;
- TPU, if the optional `torch_xla` package is installed: `torch_xla.core.xla_model.set_rng_state`.

Seed a device only when it is available. Also switch cuDNN to deterministic mode with `torch.backends.cudnn.deterministic = True`. This makes repeated runs much easier to compare, especially while debugging models and tests.

</details>


In [ ]:
# Initializing random state generators
# Edit: ../../source/code/src/utils/seed.py
src.utils.seed.seed_all(0)
# ^^^ This function should be implemented

unittests.check_seed_all(answer)


In [ ]:
# Checking the implementation
course_utils.checker.print_progress(
    'Task 0: Prepare the environment', answer,
    prefixes=('seed_check',),
    is_small_run=IS_UNITTEST)


# Training a Fully Connected Feedforward Neural Network

<details>
<summary>Details</summary>

## Overview

In this exercise, we will train a fully connected neural network (FCNN) to classify MNIST digits into 10 classes: 0, 1, 2, ..., 9.

## Ingredients

To build and train this model, we will use:
- a fully connected neural network;
- the MNIST dataset;
- `AdamW` optimization;
- accuracy as a metric;
- cross-entropy loss, which internally combines `LogSoftmax` and negative log-likelihood.

## Why This Matters

The important part is not only the final accuracy. In this notebook, we will assemble the basic pieces that appear in almost every training pipeline: dataset, model, loss, optimizer, dataloaders, training loop, validation loop, and simple diagnostics.

</details>


# Task 1: Prepare the Dataset class

<details>
<summary>What to do</summary>

## Goal

In this task, you will implement your first dataset class for one of the most popular machine-learning datasets: MNIST.

Fill in [src/datasets/mnist_simple.py](../../source/code/src/datasets/mnist_simple.py) so that `MNISTSimpleDataset` works as a PyTorch-style dataset. The file should open next to this notebook.

## Methods To Implement

You need to implement three methods:
- `__init__`, the constructor;
- `__len__`, used by `len(dataset)`;
- `__getitem__`, used by `dataset[index]`.

You will use this same pattern for your own datasets later. Real datasets will often need more complicated loading and preprocessing logic, but the interface will stay the same.

## Implementation Details

- In `__init__`, load MNIST with `mnist_class(root='~/', train=train, download=True)`. `mnist_class` is `torchvision.datasets.MNIST`; in the small run (`small=True`) it is `SmallMNIST`, a tiny offline stand-in with the same `data` and `targets` fields. Use the `train` argument to decide whether to load the training or validation split. Store the raw images in `self.X` and labels in `self.y`. Store the data under `~/` so that downloaded files do not clutter the repository.
- In `__len__`, return the number of samples in the dataset.
- In `__getitem__(self, index)`, return a dictionary with two keys:
    - `image`: the selected image converted to `float32` and scaled from `[0, 255]` to `[-1, 1]`;
    - `label`: the selected label converted to `long`.

## Why Scaling Helps

Scaling the image values to `[-1, 1]` keeps the inputs numerically balanced around zero. This usually makes optimization easier than feeding raw pixel values in `[0, 255]`.

</details>


In [ ]:
# Edit: ../../source/code/src/datasets/mnist_simple.py
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
# ^^^ This class should be implemented in this task

Check that the data is prepared:

In [ ]:
unittests.check_dataset(answer, MNIST_train, MNIST_valid)


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Prepare the Dataset class', answer,
    prefixes=('X_train', 'X_valid', 'y_train', 'y_valid'),
    is_small_run=IS_UNITTEST)


Neural networks are usually trained with floating-point tensors, most commonly `float32` or `float16`. In this exercise, the dataset class should return images as `float32` tensors.


Let us visualize one training sample.


In [ ]:
import matplotlib.pyplot as plt
train_sample = MNIST_train[0]
plt.imshow(train_sample['image'])
plt.show()
print(train_sample['label'])


## Batch Updating Protocol

<details>
<summary>Details</summary>

## Motivation

Before we build the model, let us agree on how information moves through the training pipeline. A dataset item starts as a small dictionary, for example `{"image": ..., "label": ...}`. When the dataloader combines many items into a mini-batch, the training loop wraps that batch into a nested dictionary:

```python
batch = {'data': original_batch}
```

## Batch Sections

The `data` section contains values that came from the dataset, such as `batch['data']['image']` and `batch['data']['label']`. The model should not overwrite these values. Instead, it adds new sections as it works:

- `batch['signals']`: raw differentiable tensors produced by the network, such as logits in `batch['signals']['output']`;
- `batch['postprocessed']`: non-loss-facing predictions or summaries, such as predicted class indices in `batch['postprocessed']['class']`;
- `batch['loss']`: the scalar loss tensor computed by the loss function during training.

## Responsibility Split

This update-in-place style lets the model, loss function, metric function, and training loop communicate through one shared object while still keeping their responsibilities separate. The model writes signals and predictions, the loss reads the batch and returns a scalar, metrics read the batch and return aggregated counts, and the training loop controls optimization.

</details>


# Task 2: Build the FCNN class

<details>
<summary>What to do</summary>

## Goal

It is time to create our neural network. Edit [src/models/feedforward/simple_fcnn.py](../../source/code/src/models/feedforward/simple_fcnn.py); it should open automatically with this chapter.

In PyTorch, trainable models usually inherit from `torch.nn.Module`. This gives the model access to parameters, submodules, `model.train()`, `model.eval()`, optimizers, hooks, device movement, and other PyTorch machinery.

## Batch Protocol

We will use the batch updating protocol described above: the model reads images from `batch['data']['image']`, writes logits to `batch['signals']['output']`, and writes predicted classes to `batch['postprocessed']['class']`.

## Parts To Implement

Implement three parts:
- `__init__`, which creates the modules of the neural network. Remember to initialize the parent `torch.nn.Module` with `super().__init__()`.
- `__forward_kernel`, which receives an image tensor, flattens it from `[batch, 28, 28]` to `[batch, 784]`, passes it through the fully connected layers, and returns raw logits. This method is also used for the model visualization.
- `forward`, which receives the nested batch dictionary, reads `batch['data']['image']`, calls `__forward_kernel`, stores logits in `batch['signals']['output']`, calls `argmax` to get predicted classes, and stores them in `batch['postprocessed']['class']`.

</details>


In [ ]:
# Constructing a network
# Edit: ../../source/code/src/models/feedforward/simple_fcnn.py
simple_network = src.models.feedforward.simple_fcnn.SimpleFCNN([28 * 28], n_classes=10)
# ^^^ This class should be implemented in this task

# Here we intialize the network so that the initial state is predictable
src.utils.deterministic_init(simple_network)

# Forward pass
batch = {'data': {'image': MNIST_train[0]['image'][None]}}
result = simple_network.forward(batch)
print("Network calling result:\n")
pprint(result)

# Network health check
unittests.check_network(answer, simple_network)


In [ ]:
course_utils.checker.print_progress(
    'Task 2: Build the FCNN class', answer,
    prefixes=('check_batch',),
    is_small_run=IS_UNITTEST)


## Model Graph

<details>
<summary>Details</summary>

Let us also draw the model graph. A visual graph is often helpful when you want to check whether the signal flows through the modules in the order you intended.

The utility [src/utils/model_visualisation.py](../../source/code/src/utils/model_visualisation.py) uses a small internal adapter so that `torchview` draws only the tensor path through `__forward_kernel`. That keeps the graph focused on the neural-network layers rather than on the nested batch dictionary and postprocessing keys.

## Empty Backbone Note

One important detail: `backbone` and `classifier` are module names, while the graph is built from the actual tensor operations. If the current model uses `channels=[784]`, then `backbone` is an empty `Sequential`, so there is no hidden-layer operation for the graph to show. In that case we draw a small non-empty example network below just for visualization.

</details>


In [ ]:
print("Network structure:")
print(simple_network)

print("Forward kernel schema:")
model_graph = src.utils.model_visualisation.draw_forward_kernel_graph(simple_network)
if not IS_UNITTEST and shutil.which("dot"):
    display(model_graph.visual_graph)
elif not IS_UNITTEST:
    print("Install the Graphviz dot executable to display this graph.")

# Task 3: Create the Loss Function, Optimizer, Scheduler, and Metric


<details>
<summary>What to do</summary>

## Goal

Now that we have prepared the dataset and the network, we need the remaining pieces for model training:
- a dictionary of batch-level loss functions;
- an optimizer;
- a learning-rate scheduler;
- a dictionary of batch-level metric functions.

## Training-Loop Contract

The training loop in [src/train_loop.py](../../source/code/src/train_loop.py) calls `model(batch)`, then evaluates every function in the `losses` dictionary and every function in the `metrics` dictionary. Therefore the loss and metric functions should read the same nested batch keys that the model writes.

## Components To Create

Use the following components:
- `loss(batch)`: define the cross-entropy loss between `batch['signals']['output']` and `batch['data']['label']`. Keep logits shaped as `[batch, n_classes]`; only flatten labels if needed.
- `losses`: create a dictionary that maps the loss name `'ce'` to the `loss` function. During training, the loop sums all named losses into one scalar for backpropagation, while keeping the individual named losses for progress bars, history, and MLflow logging.
- `accuracy(batch)`: define accuracy, compare `batch['postprocessed']['class']` with `batch['data']['label']`, and return `(enumerator, denominator)` so the loop can aggregate the metric across batches. Enumerator should be the number of correct labels, while denominator should be the number of total samples in the batch.
- `metrics`: create a dictionary that maps the metric name `'accuracy'` to the `accuracy` function. The dictionary keys are used in MLflow metric names and in the returned history.
- `torch.optim.AdamW` with learning rate `1.0e-3` as the optimizer and it should be targeted on the model's parameters (get them with `model.parameters()`).
- `torch.optim.lr_scheduler.ReduceLROnPlateau` as the scheduler on top of the `optimizer`.

## Scope

Create these objects and functions in the cell below. Accuracy is the only metric in this task, and cross-entropy is the only loss, but the loop accepts any number of named losses and metrics that follow these conventions.

</details>


In [ ]:
# Edit this cell in task.ipynb.
import torchmetrics

def loss(batch):
    result = batch['signals']['output'].abs().sum()
    ## YOUR CODE HERE
    return result

losses = {'ce': loss}

def accuracy(batch):
    enum, denom = 0, 1
    ## YOUR CODE HERE
    return enum, denom

metrics = {'accuracy': accuracy}
optimizer = torch.optim.SGD(simple_network.parameters(), lr=0.0)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1)

## YOUR CODE HERE
# Build a correct optimizer and a correct scheduler for that optimizer



In [ ]:
unittests.check_training_setup(answer, losses, metrics, optimizer, scheduler)


In [ ]:
course_utils.checker.print_progress(
    'Task 3: Create the Loss Function, Optimizer, Scheduler, and Metric', answer,
    prefixes=(
        'loss_names',
        'metric_names',
        'optimizer_',
        'scheduler_',
        'test_loss_val',
        'test_metrics_val',
    ),
    is_small_run=IS_UNITTEST)


# Task 4: Create DataLoaders

<details>
<summary>What to do</summary>

## Goal

Now it is time to start the training process.

First, turn the datasets into dataloaders. A dataloader creates mini-batches and can shuffle the data for us.

## Implementation Details

Use `torch.utils.data.DataLoader` to create training and validation dataloaders.

Important details:
- shuffle the training data;
- do not shuffle the validation data;
- use `drop_last=True` for the training dataloader so that all training batches have the same size;
- use the `batch_size` equal to 32

</details>


In [ ]:
# Edit this cell in task.ipynb.
from tqdm import trange, tqdm
import os

batch_size = 8
train_dl = DataLoader(MNIST_train, batch_size=batch_size, shuffle=False)
valid_dl = DataLoader(MNIST_valid, batch_size=batch_size, shuffle=False)

# ^^^ These objects should be tuned up in this task, please consult with the documentation

unittests.check_dataloaders(answer, train_dl, valid_dl)


In [ ]:
course_utils.checker.print_progress(
    'Task 4: Create DataLoaders', answer,
    prefixes=('train_batch', 'valid_batch'),
    is_small_run=IS_UNITTEST)


# Task 5: Create training loop

<details>
<summary>What to do</summary>

## Goal

First, let us outline the training loop. Your task is to complete the placeholder blocks in [src/train_loop.py](../../source/code/src/train_loop.py), including the small helper functions for named losses/metrics and the training/validation blocks inside `train_model`. This notebook prepares the model, data, loss dictionary, metric dictionary, optimizer, scheduler, and logger, then calls that function.

The goal of `train_model` is to train the network for `n_epochs`, evaluate it on the validation dataloader after each epoch, and return a `history` dictionary with averaged losses and metrics. The surrounding code in `src/train_loop.py` already creates the history dictionaries, accumulators, MLflow logging calls, scheduler step, and return value. You only need to fill in the `## YOUR CODE HERE` regions; the surrounding progress bars, history dictionaries, scheduler step, MLflow logging, and return value are already wired up.

## Where You Will Work

You will work in two places:
1. In this notebook, define `losses`, `metrics`, `optimizer`, and `scheduler`.
2. In `src/train_loop.py`, complete the placeholder blocks that compute named losses, sum them for backpropagation, update named metrics, and run train/validation steps.

## Experiment Logging

Real projects often log experiments with tools such as MLflow, TensorBoard, Weights & Biases, or Neptune. In this notebook, we will use MLflow because it can log runs locally without requiring an external account.

## Training Stages

Training usually runs for several epochs. Each epoch has two stages:
- training;
- validation.

## Batch Protocol

For every dataloader batch, the loop first wraps it into the course batch protocol:
```python
batch = {'data': original_batch}
```
Then the model reads `batch['data']`, writes logits to `batch['signals']['output']`, and writes class predictions to `batch['postprocessed']['class']`.

## Helper Placeholders

Before the main loop, implement the helper placeholders:
- `_compute_losses(losses, batch)` should return a dictionary with the same keys as `losses`, where each value is the result of applying that loss function to the batch.
- `_sum_losses(batch_losses)` should add all values in the named-loss dictionary into one scalar tensor used for backpropagation; the individual named losses remain available for progress bars, history, and MLflow logging.
- `_update_metric_totals(metric_totals, metrics, batch)` should call each metric function, receive `(enumerator, denominator)`, and add those values to the matching accumulator.

## Training Placeholder

In the training-stage placeholder, implement one mini-batch update. The code should:
1. Switch the model to training mode with `model.train()`.
2. Reset gradients with `optimizer.zero_grad()`.
3. Run `model(batch)`.
4. Compute all named losses with `_compute_losses(losses, batch)`.
5. Sum the named losses with `_sum_losses(loss_values)` and store the result in `batch['loss']`. Store the full dictionary in `batch['losses']`.
6. Run backpropagation with `loss_value.backward()`.
7. Update model parameters with `optimizer.step()`.
8. Update every metric in the `metrics` dictionary by calling `_update_metric_totals(train_metrics, metrics, batch)`. The surrounding code uses the named losses for history, progress bars, and MLflow logging.

## Validation Placeholder

In the validation-stage placeholder, implement one validation mini-batch pass. The code already runs inside `with torch.no_grad():`. It should switch the model to evaluation mode, run `model(valid_batch)`, compute all named validation losses, store `valid_batch['losses']` and the summed `valid_batch['loss']`, and update every metric in the `metrics` dictionary. Do not call `backward()` or `optimizer.step()` during validation.

## Epoch Outputs

At the end of each epoch, the loop appends averaged values to a history dictionary with keys `train_loss`, `valid_loss`, `train_metrics`, and `valid_metrics`; each value is itself a dictionary keyed by loss or metric name. If a scheduler is provided, it is stepped once per epoch. If an MLflow logger is provided, the loop logs per-batch named training losses, per-epoch named losses/metrics, and training curves.

</details>


In [ ]:
n_epochs = run_params.TRAINING_EPOCHS

training_logger = src.utils.mlflow.prepare_mlflow_logger('simple_network_training')

training_history = src.train_loop.train_model(
    simple_network,
    n_epochs,
    train_dl,
    valid_dl,
    losses,
    optimizer,
    metrics=metrics,
    scheduler=scheduler,
    mlflow_logger=training_logger,
    run_name='simple_network_training',
    device=device)

training_logger.end()


In [ ]:
unittests.check_training(answer, training_history)


In [ ]:
course_utils.checker.print_progress(
    'Task 5: Create training loop', answer,
    prefixes=(
        'train_loss_history',
        'valid_loss_history',
        'train_acc_history',
        'valid_acc_history',
    ),
    is_small_run=IS_UNITTEST)


# Task 6: Experiment Time


<details>
<summary>What to do</summary>

## Goal

Now we have everything needed to train a network.

Create and train several models:
- a 1-layer FCNN, which is multiclass logistic regression;
- a 2-layer FCNN;
- a 3-layer FCNN.

## Epoch Count

The experiments train only in the full run (`IS_UNITTEST = False`), `FULL_RUN['EXPERIMENT_EPOCHS']` epochs each (100 by default), set in the first cell. The small run only builds the networks.

</details>


In [ ]:
# Edit this cell in task.ipynb.
def run_fcnn_experiment(run_name, model, n_epochs=run_params.EXPERIMENT_EPOCHS):
    experiment_optimizer = torch.optim.AdamW(model.parameters(), lr=1.0e-3)
    experiment_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(experiment_optimizer)
    experiment_logger = src.utils.mlflow.prepare_mlflow_logger(run_name)
    history = src.train_loop.train_model(
        model,
        n_epochs,
        train_dl,
        valid_dl,
        losses,
        experiment_optimizer,
        metrics=metrics,
        scheduler=experiment_scheduler,
        mlflow_logger=experiment_logger,
        run_name=run_name,
        device=device)
    experiment_logger.end()
    return history

one_layer_fcnn = src.models.feedforward.simple_fcnn.SimpleFCNN(
    [28 * 28],
    n_classes=10,
    activation=torch.nn.LeakyReLU)

## YOUR CODE HERE
# Create the 2-layer and the 3-layer networks


In [ ]:
# Edit this cell in task.ipynb.
# The experiments train only in the full run (IS_UNITTEST = False).
if not IS_UNITTEST:
    one_layer_history = run_fcnn_experiment('one_layer_fcnn', one_layer_fcnn)
    ## YOUR CODE HERE
    # Train the 2-layer and the 3-layer networks


# Task 7: Extract the First-Layer Weights


<details>
<summary>What to do</summary>

## Goal

Look at what the first layer of each network has learned. Every row of the first linear layer's weight matrix has `28 * 28` numbers, one per pixel, so it can be shown as a `28 x 28` image.

## Implementation Details

Implement `first_weight_matrix(model)` in the cell below:
- go through the submodules of the network with `model.modules()`, in order;
- return the `weight` of the first `torch.nn.Linear` you meet; its shape is `[out_features, in_features]`, here `[n_units, 784]`;
- if the network has no linear layer, raise `ValueError`.

The next cell turns the rows of this matrix into images and plots them for the networks of Task 6.

</details>


In [ ]:
# Edit this cell in task.ipynb.
def first_weight_matrix(model):
    weights = torch.zeros(10, 28 * 28)
    ## YOUR CODE HERE
    # Return the weight of the first torch.nn.Linear in model.modules()
    return weights


In [ ]:
def plot_first_layer_weights(model, title, logger=None, max_images=10):
    weights = first_weight_matrix(model).detach().cpu()
    if weights.shape[1] != 28 * 28:
        raise ValueError('The first linear layer cannot be reshaped into 28 x 28 images.')

    n_images = min(max_images, weights.shape[0])
    n_cols = min(5, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 2.2 * n_rows))
    axes = np.asarray(axes).reshape(-1)

    for index in range(n_images):
        axes[index].imshow(weights[index].reshape(28, 28), cmap='grey')
        axes[index].set_title(f'unit {index}')
        axes[index].axis('off')

    for index in range(n_images, len(axes)):
        axes[index].axis('off')

    fig.suptitle(title)
    fig.tight_layout()

    if logger is not None:
        logger.log_figure(fig, 'figures/first_layer_weights.png')
    plt.show()
    plt.close(fig)
    # return fig


In [ ]:
plot_first_layer_weights(one_layer_fcnn, 'one_layer_fcnn: classifier weights')


In [ ]:
# plot_first_layer_weights(two_layer_fcnn, 'two layer fcnn: stems')

In [ ]:
# plot_first_layer_weights(three_layer_fcnn, 'three layer fcnn: stems')

# Time to Think

<details>
<summary>Details</summary>

## Convergence

What converges faster?
- 1-layer NN, also known here as multiclass logistic regression;
- 2-layer NN;
- 3-layer NN.

Would you expect a deeper network, for example 10 layers, to train better or worse with this simple setup?

## Interpreting Learned Patterns

What do the patterns from the 1-layer model resemble? Should they look like digit templates?

What do the first-layer patterns from the 2-layer and 3-layer models resemble? Should they be as directly interpretable as the 1-layer weights?

</details>


# Task 8 (Optional): More Experiments

<details>
<summary>What to do</summary>

## Activation Functions

Try training the networks above with different activation functions. Focus on the 2-layer network first:
- `Sigmoid`
- `Tanh`
- `ReLU`
- `LeakyReLU` <- good default option
- `SiLU` (also known as Swish)
- `GELU` (Gaussian Linear Unit, usually used in LLMs)

## Optimizers

Try different optimizers:
- `AdamW` <- this optimizer is usually considered as a baseline optimizer with learning rate `1.0e-3 -- 1.0e-4`
- `SGD` <- classical, but tricky one

Which optimizer is easier to tune?

## Longer Runs And Batch Sizes

If you have access to Google Colab or another machine with a GPU, try training for longer. Also try different batch sizes. Batch sizes are often selected as powers of 2:
- 2
- 4
- 8
- 16
- 32 <- this batch is usually considered as good enough, but still increasing it won't hurt
- 64
- etc.

## Discussion

Share your best recipe and discuss what improved your results.

</details>
